After running this notebook final output: 
```
data_CLB = 
{
    "CP1A":{
        "data":(16 * 2, 18000, 10, 3)
        "drug":[CP, V, ...] * 32,
        "concentration":[0.01, 0] * 32,
     } 
     "CP1B": {
        "data":(24 *2, 18000, 10, 3)
        "drug":[CP, V, ...] * 48,
        "concentration":[0.01, 0] * 48,
     }
     "INH1": {
        "data":(27, 18000, 10, 3)
        "drug":[CP, V, ...] * 27,
        "concentration":[],
     }
     "INH2": {
         "data":(72, 18000, 10, 3)
         "drug":[AM, V, PF, MJ, AMPF, AMMJ...],
         "concentration":[] * 72,
      }
      "MOS1aD": {
         "data":(24, 18000, 10, 3)
         "drug":[...] * 24,
         "concentration":[] * 24,
      }
}
```

In [2]:
# import dependencies
import os
import pickle
import numpy as np
from pathlib import Path

In [3]:
label_map = {
    "CP1A":{
        "A": {"drug":"V", "concentration":0},
        "B": {"drug":"CP", "concentration":0.03},
        "C": {"drug":"CP", "concentration":0.01},
        "D": {"drug":"CP", "concentration":0.3},
    },
    "CP1B":{
        "A": {"drug":"V", "concentration":0},
        "B": {"drug":"CP", "concentration":0.03},
        "C": {"drug":"CP", "concentration":0.01},
        "D": {"drug":"CP", "concentration":0.3},
    }, 
    "INH1":{
        "A": {"drug":"V", "concentration":0},
        "B": {"drug":"PF", "concentration":30},
        "C": {"drug":"MJ", "concentration":2.5},
    },
    "INH2":{
        "A": {"drug":"V", "concentration":0},
        "B": {"drug":"AM", "concentration":3},
        "C": {"drug":"PF", "concentration":10},
        "D": {"drug":"MJ", "concentration":1.25},
        "E": {"drug":"AMPF", "concentration":[3, 10]},
        "F": {"drug":"AMMJ", "concentration":[3, 1.25]},
    }, 
    "MOS1aD":{
        "1": {"drug":"V", "concentration":0},
        "2": {"drug":"V", "concentration":0},
        "B": {"drug":"CP", "concentration":0.3},
        "C": {"drug":"PF", "concentration":30},
        "D": {"drug":"MJ", "concentration":2.5},
        "E": {"drug":"H", "concentration":20},
    }
}

# CLB

## Data

In [4]:
path_to_data_dir = "/home/rguo_hpc/myfolder/code/mocap/data/"
datasets = [
            #"CP1A", "CP1B",  # Wait until cvg works
            "INH1", "INH2", "MOS1aD"
            ]
sample_frequency = 5 

data_CLB = {key:{} for key in datasets} # Store processed data for final output

In [5]:
for dataset in datasets: 
    
    print(dataset)
    filepath = Path(path_to_data_dir + dataset + "_CLB.pkl")
    data = []
    drug_label = []
    concentration_label = []
    # 0. load data
    with open(filepath, 'rb') as file:  # Read pickle file 
        raw_data = pickle.load(file)
    
    for seq_ix, (seq_name, sequence) in enumerate(raw_data.items()):
        # 1. Sample 1/5 data
        data.append(sequence["data"][::sample_frequency])
        
        # 2.  process label
        dataset_label_map = label_map[dataset]
        label = sequence["label"]
        drug_label.append(dataset_label_map[label]["drug"])
        concentration_label.append(dataset_label_map[label]["concentration"])
    
    data_CLB[dataset]["data"] = np.array(data, dtype=np.float32)
    data_CLB[dataset]["drug"] = drug_label
    data_CLB[dataset]["concentration"] = concentration_label
        

INH1
INH2
MOS1aD


In [14]:
with open("/home/rguo_hpc/myfolder/code/mocap/data/data_CLB.pkl", 'wb') as file:
    pickle.dump(data_CLB, file)

In [46]:
import pickle

# Read data
with open('/home/rguo_hpc/myfolder/code/mocap/data/data_CLB.pkl', 'rb') as file:
    data_CLB = pickle.load(file)

In [59]:
import numpy as np
data = data_CLB["INH1"]["data"]

In [ ]:
for i in range(len(data)):
    missing_per_sample = np.isnan(data[i]).sum(axis=(1, 2))
    print(len(np.where(missing_per_sample>3)[0]))

In [78]:
import numpy as np

n_total = 17
n_samples = 7

indices = np.linspace(0, n_total - 1, n_samples, dtype=int)
print(indices)

[ 0  2  5  8 10 13 16]


## Label

In [5]:
drug = []
for dataset_name in ["INH1", "INH2", "MOS1aD"]:
    drug = drug + data_CLB[dataset_name]["drug"]

In [10]:
len(set(drug))

8

In [11]:
from collections import Counter
Counter(drug)

Counter({'V': 34,
         'MJ': 21,
         'PF': 21,
         'AM': 12,
         'AMMJ': 12,
         'AMPF': 12,
         'H': 6,
         'CP': 5})

# FL2